# Progetto: Analisi ed Evoluzione della Serie A (1993 - 2026)
### Corso di Principi e Modelli della Percezione

**Gruppo di lavoro:**
- Diego Cozzi (51299A)
- Mario Rovagnati (50644A)
- Matteo Dini (51073A)

---

### Pipeline e Struttura del Notebook:
1. **Sezione 1:** Download e pulizia dataset storico da `football-data.co.uk` (1993/94 – 2025/26, 11.726 incontri).
2. **Sezione 2:** Estrazione tabelle aggregate per le slide storiche (`dati_slide7_tiri.csv`, `dati_slide8_classifica_storica.csv`, `dati_slide9_forbice.csv`).
3. **Sezione 3:** Consolidamento dati decennali 10 club Understat 2014–2024 (`dati_gol_reali_10_squadre.csv`, `dati_scostamento_gol_xg.csv`).
4. **Sezioni Grafiche (4-10):** Generazione ed esportazione di tutti i grafici della presentazione in duplice versione:
   - **Versione `_con_guida.png`**: con titoli, etichette e annotazioni esplicative.
   - **Versione `_clean.png`**: versione muta/pulita ad alta risoluzione per il montaggio finale nelle slide.

---
## Sezione 1: Download e Consolidamento Dataset Storico (1993 - 2026)
Scarica ed elabora i file CSV delle 33 stagioni di Serie A, gestisce campi mancanti e normalizza la nomenclatura.

In [ ]:
import io, csv, requests
import pandas as pd
import numpy as np

colonne_target = [
    "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR",
    "HS", "AS", "HST", "AST", "HF", "AF", "HY", "AY", "HR", "AR"
]
colonne_numeriche = ["FTHG", "FTAG", "HS", "AS", "HST", "AST", "HF", "AF", "HY", "AY", "HR", "AR"]

frames = []
for start in range(1993, 2026):
    code = f"{str(start)[2:]}{str(start+1)[2:]}"
    url = f"https://www.football-data.co.uk/mmz4281/{code}/I1.csv"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        testo = r.content.decode("latin-1").replace("\x00", "")
        righe = list(csv.reader(io.StringIO(testo)))
        intestazione = [h.strip().lstrip("\ufeff") for h in righe[0]]
        pos = {c: intestazione.index(c) for c in colonne_target if c in intestazione}

        records = []
        for riga in righe[1:]:
            i_casa = pos.get("HomeTeam")
            if i_casa is None or len(riga) <= i_casa or not riga[i_casa].strip():
                continue
            records.append({c: (riga[i] if i < len(riga) else None) for c, i in pos.items()})

        df_stag = pd.DataFrame(records)
        df_stag["stagione"] = f"{start}/{start+1}"
        frames.append(df_stag)
    except Exception as e:
        pass

matches = pd.concat(frames, ignore_index=True)
num_effettivi = [c for c in colonne_numeriche if c in matches.columns]
matches[num_effettivi] = matches[num_effettivi].apply(pd.to_numeric, errors="coerce")

diz_traduzione = {
    "Date": "data", "HomeTeam": "squadra_casa", "AwayTeam": "squadra_trasferta",
    "FTHG": "gol_casa", "FTAG": "gol_trasferta", "FTR": "esito",
    "HS": "tiri_casa", "AS": "tiri_trasferta", "HST": "tiri_porta_casa", "AST": "tiri_porta_trasferta",
    "HF": "falli_casa", "AF": "falli_trasferta", "HY": "gialli_casa", "AY": "gialli_trasferta",
    "HR": "rossi_casa", "AR": "rossi_trasferta"
}
matches = matches.rename(columns=diz_traduzione)
matches["esito"] = matches["esito"].map({"H": "casa", "D": "pareggio", "A": "trasferta"})

matches.to_csv("serie_a_partite.csv", index=False)
print(f"Dataset 'serie_a_partite.csv' salvato con successo: {len(matches)} partite totali.")



Dataset 'serie_a_partite.csv' salvato con successo: 11726 partite totali.


---
## Sezione 2: Calcolo Dati Aggregati per le Slide Storiche (Slide 7, 8 e 9)
Estrae i dataset sintetici necessari per le analisi storiche: tiri, classifica normalizzata a 3 punti per vittoria e forbice competitiva.

In [ ]:
# 1. Slide 7: Volume e precisione tiri dal 2005/06
df_tiri = matches.dropna(subset=["tiri_casa", "tiri_trasferta"]).copy()
df_tiri["tiri_totali"] = df_tiri["tiri_casa"] + df_tiri["tiri_trasferta"]
df_tiri["tiri_porta_totali"] = df_tiri["tiri_porta_casa"] + df_tiri["tiri_porta_trasferta"]

tiri_per_stagione = df_tiri.groupby("stagione").agg(
    partite=("data", "count"),
    tiri_medi=("tiri_totali", "mean"),
    tiri_porta_medi=("tiri_porta_totali", "mean")
).round(2).reset_index()
tiri_per_stagione["precisione_%"] = round((tiri_per_stagione["tiri_porta_medi"] / tiri_per_stagione["tiri_medi"]) * 100, 1)
tiri_per_stagione.to_csv("dati_slide7_tiri.csv", index=False)

# 2. Slide 8: Classifica storica standardizzata a 3 punti a vittoria (33 stagioni)
records_pt = []
for _, r in matches.iterrows():
    st = r["stagione"]
    if r["esito"] == "casa":
        records_pt.extend([{"stagione": st, "squadra": r["squadra_casa"], "punti": 3},
                          {"stagione": st, "squadra": r["squadra_trasferta"], "punti": 0}])
    elif r["esito"] == "trasferta":
        records_pt.extend([{"stagione": st, "squadra": r["squadra_casa"], "punti": 0},
                          {"stagione": st, "squadra": r["squadra_trasferta"], "punti": 3}])
    else:
        records_pt.extend([{"stagione": st, "squadra": r["squadra_casa"], "punti": 1},
                          {"stagione": st, "squadra": r["squadra_trasferta"], "punti": 1}])

df_punti = pd.DataFrame(records_pt)
classifiche = df_punti.groupby(["stagione", "squadra"])["punti"].sum().reset_index()
classifica_storica = classifiche.groupby("squadra").agg(
    punti_totali=("punti", "sum"),
    stagioni_disputate=("stagione", "nunique")
).sort_values("punti_totali", ascending=False).reset_index()
classifica_storica["media_punti_stagione"] = round(classifica_storica["punti_totali"] / classifica_storica["stagioni_disputate"], 1)
classifica_storica.head(10).to_csv("dati_slide8_classifica_storica.csv", index=False)

# 3. Slide 9: Forbice competitiva 1ª classificata vs quota salvezza
forbice_list = []
for stagione, gruppo in classifiche.groupby("stagione"):
    grp_sorted = gruppo.sort_values("punti", ascending=False).reset_index(drop=True)
    punti_1 = grp_sorted.iloc[0]["punti"]
    idx_salv = 16 if len(grp_sorted) >= 20 else 14
    punti_salv = grp_sorted.iloc[idx_salv]["punti"]
    forbice_list.append({
        "stagione": stagione,
        "punti_campione": punti_1,
        "punti_salvezza": punti_salv,
        "gap": punti_1 - punti_salv
    })
df_forbice = pd.DataFrame(forbice_list)
df_forbice.to_csv("dati_slide9_forbice.csv", index=False)
print("Aggregati per Slide 7, 8 e 9 salvati con successo.")



Aggregati per Slide 7, 8 e 9 salvati con successo.


---
## Sezione 3: Consolidamento Dati Decennali Squadre (Understat 2014–2024)
Archiviazione dei dati certificati sulle 10 squadre costantemente presenti nella massima divisione (380 match): gol effettivi realizzati e gol attesi (xG).

In [ ]:
stagioni_10 = ["14/15", "15/16", "16/17", "17/18", "18/19", "19/20", "20/21", "21/22", "22/23", "23/24"]
squadre = ["Napoli", "Inter", "Juventus", "Atalanta", "Roma", "Lazio", "Milan", "Fiorentina", "Sassuolo", "Torino"]

gol_reali = {
    "Napoli":     [70, 80, 94, 77, 74, 61, 86, 74, 77, 55],
    "Inter":      [59, 50, 72, 66, 57, 81, 89, 84, 71, 89],
    "Juventus":   [72, 75, 77, 86, 70, 76, 77, 57, 56, 54],
    "Atalanta":   [38, 41, 62, 57, 77, 98, 90, 65, 66, 72],
    "Roma":       [54, 83, 90, 61, 66, 77, 68, 59, 50, 65],
    "Lazio":      [71, 52, 74, 89, 56, 79, 61, 77, 60, 49],
    "Milan":      [56, 49, 57, 56, 55, 63, 74, 69, 64, 76],
    "Fiorentina": [61, 60, 63, 54, 47, 51, 47, 59, 53, 61],
    "Sassuolo":   [49, 49, 58, 29, 53, 69, 64, 64, 47, 43],
    "Torino":     [48, 52, 71, 54, 52, 44, 50, 46, 42, 36]
}

xg_dict = {
    "Napoli":     [66, 74, 82, 73, 71, 65, 80, 68, 70, 56],
    "Inter":      [55, 48, 68, 63, 59, 75, 82, 79, 68, 83],
    "Juventus":   [64, 69, 71, 74, 66, 73, 71, 55, 57, 54],
    "Atalanta":   [41, 44, 58, 56, 74, 91, 84, 67, 65, 68],
    "Roma":       [56, 76, 82, 63, 64, 72, 69, 62, 53, 62],
    "Lazio":      [63, 51, 67, 78, 58, 71, 59, 69, 57, 47],
    "Milan":      [52, 48, 54, 55, 54, 61, 68, 66, 62, 71],
    "Fiorentina": [58, 61, 62, 56, 52, 54, 52, 62, 57, 59],
    "Sassuolo":   [46, 47, 53, 35, 51, 63, 61, 60, 48, 46],
    "Torino":     [46, 49, 64, 53, 50, 47, 52, 45, 43, 39]
}

diff_dict = {sq: [g - x for g, x in zip(gol_reali[sq], xg_dict[sq])] for sq in squadre}
df_diff = pd.DataFrame(diff_dict, index=stagioni_10).T
df_gol_reali = pd.DataFrame(gol_reali, index=stagioni_10).T

df_gol_reali.to_csv("dati_gol_reali_10_squadre.csv")
df_diff.to_csv("dati_scostamento_gol_xg.csv")
print("Dataset Understat decennali 2014-2024 salvati in CSV.")



Dataset Understat decennali 2014-2024 salvati in CSV.


---
## Sezione 4: Grafico 1 — Dinamica Storica dei Gol a Partita (Slide 5)
Tracciato continuo trentennale con fascia del boom offensivo (2016–2022) e media storica. Esporta `g1_gol_con_guida.png` e `g1_gol_clean.png`.

In [ ]:
import matplotlib.pyplot as plt

matches["gol"] = matches["gol_casa"] + matches["gol_trasferta"]
media_gol = matches.groupby("stagione")["gol"].mean()
stagioni = list(media_gol.index)
valori = list(media_gol.values)
x = np.arange(len(valori))

i_start = stagioni.index("2016/2017")
i_end = stagioni.index("2021/2022")
mean_val = float(np.mean(valori))
idx_max = int(np.argmax(valori))

# Versione con Guida
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=250, facecolor="#ffffff")
ax.set_facecolor("#ffffff")
ax.axvspan(i_start, i_end, color="#eff6ff", alpha=0.8, zorder=1)
ax.fill_between(x, valori, 2.2, color="#0f172a", alpha=0.04, zorder=2)
ax.axhline(mean_val, color="#94a3b8", linestyle=":", linewidth=1.2, zorder=2)
ax.text(12, mean_val + 0.05, f"Media storica ({mean_val:.2f})", color="#64748b", fontsize=9.5, fontweight="600")

ax.plot(x, valori, color="#0f172a", linewidth=2.5, marker="o", markersize=4, zorder=3)
ax.scatter(idx_max, valori[idx_max], color="#ef4444", s=90, zorder=6, edgecolor="#ffffff", linewidth=2)
ax.annotate(f"Record storico\n{valori[idx_max]:.2f} (2020/21)",
            xy=(idx_max, valori[idx_max]), xytext=(0, 22), textcoords="offset points",
            ha="center", fontsize=9.5, fontweight="bold", color="#ef4444", zorder=7,
            arrowprops=dict(arrowstyle="->", color="#ef4444", lw=1.3))

ax.scatter(x[-1], valori[-1], color="#0f172a", s=65, zorder=6)
ax.text(x[-1], valori[-1] - 0.10, f"Oggi: {valori[-1]:.2f}", color="#0f172a", fontsize=9.5, fontweight="bold", ha="center")
ax.text((i_start + i_end) / 2, 3.25, "FASE OFFENSIVA", color="#3b82f6", fontsize=9.5, fontweight="bold", ha="center")

ax.set_ylim(2.2, 3.38)
ax.set_xlim(-0.5, len(x) - 0.5)
ax.set_xticks(x[::5])
ax.set_xticklabels([stagioni[i][:4] for i in x[::5]], fontsize=9.5, color="#64748b")
ax.set_yticks([2.4, 2.7, 3.0])
ax.set_yticklabels(["2.4", "2.7", "3.0"], fontsize=9.5, color="#64748b")
ax.set_ylabel("Gol a partita", fontsize=10.5, fontweight="600", color="#334155")
for s in ["top", "right", "left"]: ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#cbd5e1")
ax.yaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("g1_gol_con_guida.png", dpi=250, bbox_inches="tight")
plt.close()

# Versione Clean (Muta)
fig2, ax2 = plt.subplots(figsize=(10, 4.8), dpi=250, facecolor="#ffffff")
ax2.set_facecolor("#ffffff")
ax2.axvspan(i_start, i_end, color="#eff6ff", alpha=0.8, zorder=1)
ax2.fill_between(x, valori, 2.2, color="#0f172a", alpha=0.04, zorder=2)
ax2.axhline(mean_val, color="#94a3b8", linestyle=":", linewidth=1.2, zorder=2)
ax2.plot(x, valori, color="#0f172a", linewidth=2.5, marker="o", markersize=4, zorder=3)
ax2.scatter(idx_max, valori[idx_max], color="#ef4444", s=90, zorder=6, edgecolor="#ffffff", linewidth=2)
ax2.scatter(x[-1], valori[-1], color="#0f172a", s=65, zorder=6)

ax2.set_ylim(2.2, 3.38)
ax2.set_xlim(-0.5, len(x) - 0.5)
ax2.set_xticks(x[::5])
ax2.set_xticklabels([stagioni[i][:4] for i in x[::5]], fontsize=9.5, color="#64748b")
ax2.set_yticks([2.4, 2.7, 3.0])
ax2.set_yticklabels(["2.4", "2.7", "3.0"], fontsize=9.5, color="#64748b")
ax2.set_ylabel("Gol a partita", fontsize=10.5, fontweight="600", color="#334155")
for s in ["top", "right", "left"]: ax2.spines[s].set_visible(False)
ax2.spines["bottom"].set_color("#cbd5e1")
ax2.yaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax2.set_axisbelow(True)
plt.tight_layout()
plt.savefig("g1_gol_clean.png", dpi=250, bbox_inches="tight")
plt.close()

print("Grafico 1 esportato: 'g1_gol_con_guida.png' e 'g1_gol_clean.png'")



Grafico 1 esportato: 'g1_gol_con_guida.png' e 'g1_gol_clean.png'


---
## Sezione 5: Grafico 2 — Evoluzione del Fattore Campo (Slide 6)
Restringimento del divario storico tra vittorie interne ed esterne dal 1993 al 2026. Esporta `g2_fattore_campo_con_guida.png` e `g2_fattore_campo_clean.png`.

In [ ]:
perc = matches.groupby("stagione")["esito"].value_counts(normalize=True).unstack() * 100
stagioni = list(perc.index)
x = np.arange(len(stagioni))

casa = perc["casa"].values
trasferta = perc["trasferta"].values
pareggio = perc["pareggio"].values

# Versione con Guida
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=250, facecolor="#ffffff")
ax.set_facecolor("#ffffff")
ax.fill_between(x, casa, trasferta, color="#eff6ff", alpha=0.8, zorder=1)
ax.plot(x, casa, color="#1d4ed8", linewidth=2.5, marker="o", markersize=3.5, zorder=3)
ax.plot(x, trasferta, color="#ea580c", linewidth=2.5, marker="o", markersize=3.5, zorder=3)
ax.plot(x, pareggio, color="#94a3b8", linewidth=1.5, linestyle=":", zorder=2)

ax.scatter(x[-1], casa[-1], color="#1d4ed8", s=65, zorder=5)
ax.scatter(x[-1], trasferta[-1], color="#ea580c", s=65, zorder=5)
ax.scatter(x[-1], pareggio[-1], color="#94a3b8", s=45, zorder=5)

ax.scatter(x[0], casa[0], color="#1d4ed8", s=50, zorder=5)
ax.scatter(x[0], trasferta[0], color="#ea580c", s=50, zorder=5)

ax.text(x[0], casa[0] + 2.5, f"{casa[0]:.0f}%", color="#1d4ed8", fontsize=9.5, fontweight="bold", ha="center")
ax.text(x[0], trasferta[0] - 3.2, f"{trasferta[0]:.0f}%", color="#ea580c", fontsize=9.5, fontweight="bold", ha="center")

ax.text(x[-1] + 0.6, casa[-1], f"Vittorie casa: {casa[-1]:.0f}%", color="#1d4ed8", fontsize=9.5, fontweight="bold", va="center")
ax.text(x[-1] + 0.6, trasferta[-1], f"Vittorie trasferta: {trasferta[-1]:.0f}%", color="#ea580c", fontsize=9.5, fontweight="bold", va="center")
ax.text(x[-1] + 0.6, pareggio[-1], f"Pareggi: {pareggio[-1]:.0f}%", color="#64748b", fontsize=9, fontweight="600", va="center")
ax.text(15, 37.5, "Restringimento del divario", color="#3b82f6", fontsize=9.5, fontweight="bold", ha="center")

ax.set_ylim(14, 58)
ax.set_xlim(-0.5, len(x) + 4.5)
ax.set_xticks(x[::5])
ax.set_xticklabels([s[:4] for s in stagioni[::5]], fontsize=9.5, color="#64748b")
ax.set_yticks([20, 30, 40, 50])
ax.set_yticklabels(["20%", "30%", "40%", "50%"], fontsize=9.5, color="#64748b")
ax.set_ylabel("% delle partite totali", fontsize=10.5, fontweight="600", color="#334155")
for s in ["top", "right", "left"]: ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#cbd5e1")
ax.yaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("g2_fattore_campo_con_guida.png", dpi=250, bbox_inches="tight")
plt.close()

# Versione Clean
fig2, ax2 = plt.subplots(figsize=(10, 4.8), dpi=250, facecolor="#ffffff")
ax2.set_facecolor("#ffffff")
ax2.fill_between(x, casa, trasferta, color="#eff6ff", alpha=0.8, zorder=1)
ax2.plot(x, casa, color="#1d4ed8", linewidth=2.5, marker="o", markersize=3.5, zorder=3)
ax2.plot(x, trasferta, color="#ea580c", linewidth=2.5, marker="o", markersize=3.5, zorder=3)
ax2.plot(x, pareggio, color="#94a3b8", linewidth=1.5, linestyle=":", zorder=2)
ax2.scatter(x[-1], casa[-1], color="#1d4ed8", s=65, zorder=5)
ax2.scatter(x[-1], trasferta[-1], color="#ea580c", s=65, zorder=5)
ax2.scatter(x[-1], pareggio[-1], color="#94a3b8", s=45, zorder=5)
ax2.scatter(x[0], casa[0], color="#1d4ed8", s=50, zorder=5)
ax2.scatter(x[0], trasferta[0], color="#ea580c", s=50, zorder=5)
ax2.set_ylim(14, 58)
ax2.set_xlim(-0.5, len(x) - 0.5)
ax2.set_xticks(x[::5])
ax2.set_xticklabels([s[:4] for s in stagioni[::5]], fontsize=9.5, color="#64748b")
ax2.set_yticks([20, 30, 40, 50])
ax2.set_yticklabels(["20%", "30%", "40%", "50%"], fontsize=9.5, color="#64748b")
ax2.set_ylabel("% delle partite totali", fontsize=10.5, fontweight="600", color="#334155")
for s in ["top", "right", "left"]: ax2.spines[s].set_visible(False)
ax2.spines["bottom"].set_color("#cbd5e1")
ax2.yaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax2.set_axisbelow(True)
plt.tight_layout()
plt.savefig("g2_fattore_campo_clean.png", dpi=250, bbox_inches="tight")
plt.close()

print("Grafico 2 esportato: 'g2_fattore_campo_con_guida.png' e 'g2_fattore_campo_clean.png'")



Grafico 2 esportato: 'g2_fattore_campo_con_guida.png' e 'g2_fattore_campo_clean.png'


---
## Sezione 6: Grafico Slide 7 — Stile di Tiro Moderno vs 2005
Evoluzione del volume di tiri per incontro e percentuale nello specchio della porta dal 2005/06 a oggi. Esporta `g_slide7_tiri_con_guida.png` e `g_slide7_tiri_clean.png`.

In [ ]:
df_tiri = pd.read_csv("dati_slide7_tiri.csv")
x_tiri = np.arange(len(df_tiri))
anni_tiri = [s[:4] for s in df_tiri["stagione"]]

# Versione con Guida
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=250, facecolor="#ffffff")
ax.set_facecolor("#ffffff")
ax.fill_between(x_tiri, df_tiri["tiri_medi"], df_tiri["tiri_porta_medi"], color="#f1f5f9", alpha=0.9)
ax.plot(x_tiri, df_tiri["tiri_medi"], color="#0f172a", linewidth=2.5, marker="o", markersize=3.5)
ax.plot(x_tiri, df_tiri["tiri_porta_medi"], color="#2563eb", linewidth=2.5, marker="o", markersize=3.5)

ax.text(x_tiri[-1] + 0.4, df_tiri["tiri_medi"].iloc[-1], f"Tiri totali: {df_tiri['tiri_medi'].iloc[-1]:.1f}",
        color="#0f172a", fontweight="bold", fontsize=9.5, va="center")
ax.text(x_tiri[-1] + 0.4, df_tiri["tiri_porta_medi"].iloc[-1], f"Nello specchio: {df_tiri['tiri_porta_medi'].iloc[-1]:.1f}",
        color="#2563eb", fontweight="bold", fontsize=9.5, va="center")
ax.text(len(x_tiri)/2, 16.5, "Precisione costante al ~33%", color="#64748b", fontsize=9.5, fontweight="bold", ha="center")

ax.set_ylim(6, 32)
ax.set_xlim(-0.5, len(x_tiri) + 4.5)
ax.set_xticks(x_tiri[::3])
ax.set_xticklabels(anni_tiri[::3], fontsize=9.5, color="#64748b")
ax.set_ylabel("Media tiri a partita", fontsize=10.5, fontweight="600", color="#334155")
for s in ["top", "right", "left"]: ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#cbd5e1")
ax.yaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("g_slide7_tiri_con_guida.png", dpi=250, bbox_inches="tight")
plt.close()

# Versione Clean
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=250, facecolor="#ffffff")
ax.set_facecolor("#ffffff")
ax.fill_between(x_tiri, df_tiri["tiri_medi"], df_tiri["tiri_porta_medi"], color="#f1f5f9", alpha=0.9)
ax.plot(x_tiri, df_tiri["tiri_medi"], color="#0f172a", linewidth=2.5, marker="o", markersize=3.5)
ax.plot(x_tiri, df_tiri["tiri_porta_medi"], color="#2563eb", linewidth=2.5, marker="o", markersize=3.5)
ax.set_ylim(6, 32)
ax.set_xlim(-0.5, len(x_tiri) - 0.5)
ax.set_xticks(x_tiri[::3])
ax.set_xticklabels(anni_tiri[::3], fontsize=9.5, color="#64748b")
ax.set_ylabel("Media tiri a partita", fontsize=10.5, fontweight="600", color="#334155")
for s in ["top", "right", "left"]: ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#cbd5e1")
ax.yaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("g_slide7_tiri_clean.png", dpi=250, bbox_inches="tight")
plt.close()

print("Grafico Slide 7 esportato: 'g_slide7_tiri_con_guida.png' e 'g_slide7_tiri_clean.png'")



Grafico Slide 7 esportato: 'g_slide7_tiri_con_guida.png' e 'g_slide7_tiri_clean.png'


---
## Sezione 7: Grafico Slide 8 — Il Monopolio dei Punti (Classifica Storica 33 Anni)
Barre orizzontali ordinate con totale punti normalizzati a 3 per vittoria sulle 33 stagioni esaminate. Esporta `g_slide8_punti_con_guida.png` e `g_slide8_punti_clean.png`.

In [ ]:
df_punti = pd.read_csv("dati_slide8_classifica_storica.csv").head(8).sort_values("punti_totali", ascending=True)
y_pt = np.arange(len(df_punti))
colori = ["#94a3b8"] * 5 + ["#1d4ed8", "#1e293b", "#0f172a"]

# Versione con Guida
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=250, facecolor="#ffffff")
ax.set_facecolor("#ffffff")
bars = ax.barh(y_pt, df_punti["punti_totali"], color=colori, height=0.6, zorder=3)
for bar, val in zip(bars, df_punti["punti_totali"]):
    ax.text(val + 30, bar.get_y() + bar.get_height()/2, f"{int(val):,} pt".replace(",", "."),
            va="center", color="#0f172a", fontsize=9.5, fontweight="bold")

ax.set_yticks(y_pt)
ax.set_yticklabels(df_punti["squadra"], fontsize=10, fontweight="bold", color="#0f172a")
ax.set_xlim(0, 2750)
ax.set_xlabel("Punti complessivi storici (Normalizzati a 3 pt per vittoria, 1993-2026)", fontsize=10, fontweight="600", color="#334155")
for s in ["top", "right", "left", "bottom"]: ax.spines[s].set_visible(False)
ax.xaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("g_slide8_punti_con_guida.png", dpi=250, bbox_inches="tight")
plt.close()

# Versione Clean
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=250, facecolor="#ffffff")
ax.set_facecolor("#ffffff")
ax.barh(y_pt, df_punti["punti_totali"], color=colori, height=0.6, zorder=3)
ax.set_yticks(y_pt)
ax.set_yticklabels(df_punti["squadra"], fontsize=10, fontweight="bold", color="#0f172a")
ax.set_xlim(0, 2550)
ax.set_xlabel("Punti complessivi storici", fontsize=10, fontweight="600", color="#334155")
for s in ["top", "right", "left", "bottom"]: ax.spines[s].set_visible(False)
ax.xaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("g_slide8_punti_clean.png", dpi=250, bbox_inches="tight")
plt.close()

print("Grafico Slide 8 esportato: 'g_slide8_punti_con_guida.png' e 'g_slide8_punti_clean.png'")



Grafico Slide 8 esportato: 'g_slide8_punti_con_guida.png' e 'g_slide8_punti_clean.png'


---
## Sezione 8: Grafico Slide 9 — La Forbice Grandi vs Piccole (Polarizzazione del Torneo)
Evoluzione del distacco in punti tra la 1ª classificata e la 17ª (quota salvezza). Esporta `g_slide9_forbice_con_guida.png` e `g_slide9_forbice_clean.png`.

In [ ]:
df_forbice = pd.read_csv("dati_slide9_forbice.csv")
x_forb = np.arange(len(df_forbice))
anni_forb = [s[:4] for s in df_forbice["stagione"]]
media_gap = df_forbice["gap"].mean()
i_max_gap = int(df_forbice["gap"].argmax())

# Versione con Guida
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=250, facecolor="#ffffff")
ax.set_facecolor("#ffffff")
ax.axhline(media_gap, color="#94a3b8", linestyle=":", linewidth=1.2)
ax.text(x_forb[0], media_gap + 1.2, f"Divario medio storico: {media_gap:.0f} pt", color="#64748b", fontsize=9.5, fontweight="600")
ax.fill_between(x_forb, df_forbice["gap"], media_gap, color="#eff6ff", alpha=0.8)
ax.plot(x_forb, df_forbice["gap"], color="#e11d48", linewidth=2.5, marker="o", markersize=3.5, zorder=3)

ax.scatter(i_max_gap, df_forbice["gap"].iloc[i_max_gap], color="#be123c", s=80, zorder=5)
ax.annotate(f"Picco distacco: {df_forbice['gap'].iloc[i_max_gap]} pt\n(2013/14)",
            xy=(i_max_gap, df_forbice["gap"].iloc[i_max_gap]), xytext=(0, 16), textcoords="offset points",
            ha="center", fontsize=9.5, fontweight="bold", color="#be123c",
            arrowprops=dict(arrowstyle="->", color="#be123c", lw=1.2))

ax.set_ylim(25, 75)
ax.set_xlim(-0.5, len(x_forb) - 0.5)
ax.set_xticks(x_forb[::5])
ax.set_xticklabels(anni_forb[::5], fontsize=9.5, color="#64748b")
ax.set_ylabel("Distacco punti (1ª vs 17ª)", fontsize=10.5, fontweight="600", color="#334155")
for s in ["top", "right", "left"]: ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#cbd5e1")
ax.yaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("g_slide9_forbice_con_guida.png", dpi=250, bbox_inches="tight")
plt.close()

# Versione Clean
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=250, facecolor="#ffffff")
ax.set_facecolor("#ffffff")
ax.axhline(media_gap, color="#94a3b8", linestyle=":", linewidth=1.2)
ax.fill_between(x_forb, df_forbice["gap"], media_gap, color="#eff6ff", alpha=0.8)
ax.plot(x_forb, df_forbice["gap"], color="#e11d48", linewidth=2.5, marker="o", markersize=3.5, zorder=3)
ax.scatter(i_max_gap, df_forbice["gap"].iloc[i_max_gap], color="#be123c", s=80, zorder=5)
ax.set_ylim(25, 75)
ax.set_xlim(-0.5, len(x_forb) - 0.5)
ax.set_xticks(x_forb[::5])
ax.set_xticklabels(anni_forb[::5], fontsize=9.5, color="#64748b")
ax.set_ylabel("Distacco punti (1ª vs 17ª)", fontsize=10.5, fontweight="600", color="#334155")
for s in ["top", "right", "left"]: ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#cbd5e1")
ax.yaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("g_slide9_forbice_clean.png", dpi=250, bbox_inches="tight")
plt.close()

print("Grafico Slide 9 esportato: 'g_slide9_forbice_con_guida.png' e 'g_slide9_forbice_clean.png'")



Grafico Slide 9 esportato: 'g_slide9_forbice_con_guida.png' e 'g_slide9_forbice_clean.png'


---
## Sezione 9: Grafici 3A e 3B — Produzione Offensiva Squadre (Small Multiples e Heatmap Gol)
Confronto dei 10 club costanti nel decennio 2014–2024: profili temporali su scala condivisa e matrice cromatica sequenziale. Esporta `g3_small_multiples_con_guida.png`, `g3_small_multiples_clean.png`, `g3_heatmap_gol_con_guida.png` e `g3_heatmap_gol_clean.png`.

In [ ]:
import seaborn as sns

# 3A. Small Multiples - Guida
fig, axes = plt.subplots(2, 5, figsize=(12, 5.2), dpi=250, sharey=True, facecolor="#ffffff")
axes = axes.flatten()
x_idx = np.arange(len(stagioni_10))

for idx, sq in enumerate(squadre):
    ax = axes[idx]
    ax.set_facecolor("#ffffff")
    y_vals = gol_reali[sq]
    m = np.mean(y_vals)
    ax.axhline(m, color="#cbd5e1", linestyle=":", linewidth=1)
    ax.plot(x_idx, y_vals, color="#1e293b", linewidth=2, marker="o", markersize=3)
    i_max = int(np.argmax(y_vals))
    ax.scatter(i_max, y_vals[i_max], color="#ef4444", s=35, zorder=4)
    ax.set_title(sq, fontsize=10, fontweight="bold", color="#0f172a", pad=5)
    ax.set_ylim(20, 105)
    ax.set_yticks([30, 60, 90])
    ax.set_yticklabels(["30", "60", "90"], fontsize=8, color="#64748b")
    ax.set_xticks([0, 5, 9])
    ax.set_xticklabels(["14/15", "19/20", "23/24"], fontsize=7.5, color="#64748b")
    for s in ["top", "right", "left"]: ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color("#cbd5e1")
    ax.yaxis.grid(True, color="#f1f5f9", linewidth=1)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("g3_small_multiples_con_guida.png", dpi=250, bbox_inches="tight")
plt.close()

# 3A. Small Multiples - Clean
fig2, axes2 = plt.subplots(2, 5, figsize=(12, 5.2), dpi=250, sharey=True, facecolor="#ffffff")
axes2 = axes2.flatten()
for idx, sq in enumerate(squadre):
    ax = axes2[idx]
    ax.set_facecolor("#ffffff")
    y_vals = gol_reali[sq]
    ax.plot(x_idx, y_vals, color="#1e293b", linewidth=2, marker="o", markersize=3)
    i_max = int(np.argmax(y_vals))
    ax.scatter(i_max, y_vals[i_max], color="#ef4444", s=35, zorder=4)
    ax.set_title(sq, fontsize=10, fontweight="bold", color="#0f172a", pad=5)
    ax.set_ylim(20, 105)
    ax.set_yticks([30, 60, 90])
    ax.set_yticklabels(["30", "60", "90"], fontsize=8, color="#64748b")
    ax.set_xticks([0, 5, 9])
    ax.set_xticklabels(["14/15", "19/20", "23/24"], fontsize=7.5, color="#64748b")
    for s in ["top", "right", "left"]: ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color("#cbd5e1")
    ax.yaxis.grid(True, color="#f1f5f9", linewidth=1)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("g3_small_multiples_clean.png", dpi=250, bbox_inches="tight")
plt.close()

# 3B. Heatmap Gol - Guida (con testo bianco sulle celle scure >= 75)
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=250, facecolor="#ffffff")
sns.heatmap(df_gol_reali, cmap="Blues", annot=True, fmt="d", cbar=False, linewidths=1.5, linecolor="#ffffff",
            ax=ax, annot_kws={"fontsize": 9, "fontweight": "bold"})
for text in ax.texts:
    val = int(text.get_text())
    text.set_color("#ffffff" if val >= 75 else "#0f172a")

ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9.5, fontweight="bold", color="#0f172a")
ax.set_xticklabels(ax.get_xticklabels(), fontsize=9, color="#475569")
ax.tick_params(length=0)
plt.title("Gol Fatti per Squadra nelle 10 Stagioni Esaminate (2014 - 2024)", fontsize=11, fontweight="bold", pad=12, color="#0f172a")
plt.tight_layout()
plt.savefig("g3_heatmap_gol_con_guida.png", dpi=250, bbox_inches="tight")
plt.close()

# 3B. Heatmap Gol - Clean
fig2, ax2 = plt.subplots(figsize=(10, 4.8), dpi=250, facecolor="#ffffff")
sns.heatmap(df_gol_reali, cmap="Blues", annot=True, fmt="d", cbar=False, linewidths=1.5, linecolor="#ffffff",
            ax=ax2, annot_kws={"fontsize": 9, "fontweight": "bold"})
for text in ax2.texts:
    val = int(text.get_text())
    text.set_color("#ffffff" if val >= 75 else "#0f172a")

ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0, fontsize=9.5, fontweight="bold", color="#0f172a")
ax2.set_xticklabels(ax2.get_xticklabels(), fontsize=9, color="#475569")
ax2.tick_params(length=0)
plt.tight_layout()
plt.savefig("g3_heatmap_gol_clean.png", dpi=250, bbox_inches="tight")
plt.close()

print("Grafici 3A e 3B (con contrasto testo corretto) esportati con successo.")



Grafici 3A e 3B (con contrasto testo corretto) esportati con successo.


---
## Sezione 10: Grafici Avanzati xG e Merito di Gioco (Slide 10, 11 e 12)
Generazione dei grafici di approfondimento tecnico:
- **Quadrante tattico (Slide 10):** xG Fatti vs xG Subiti a partita (`g_slide10_quadrante_con_guida.png` e `_clean.png`).
- **Heatmap scostamento annuale (Slide 11):** Gol fatti meno Gol attesi per anno (`g4_heatmap_scostamento_con_guida.png` e `_clean.png`).
- **Dumbbell decennale (Slide 11):** Totale decennale Cinici vs Spreconi (`g5_dumbbell_totale_con_guida.png` e `_clean.png`).
- **Caso Studio (Slide 12):** Classifica reale vs Punti attesi xPTS stagione 2017/18 (`g_slide12_caso_studio_con_guida.png` e `_clean.png`).

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# 1. Slide 10: Quadrante Tattico xG Fatti vs Subiti
squadre_10 = ["Napoli", "Inter", "Juventus", "Atalanta", "Roma", "Lazio", "Milan", "Fiorentina", "Sassuolo", "Torino"]
xg_fatti_gara = np.array([1.89, 1.86, 1.74, 1.85, 1.78, 1.64, 1.61, 1.47, 1.31, 1.28])
xg_subiti_gara = np.array([1.02, 1.05, 0.94, 1.18, 1.14, 1.25, 1.21, 1.19, 1.52, 1.38])
med_fatti = np.median(xg_fatti_gara)
med_subiti = np.median(xg_subiti_gara)

# Quadrante - Guida (Coerente con blu/arancione)
fig, ax = plt.subplots(figsize=(10, 5), dpi=250, facecolor="#ffffff")
ax.set_facecolor("#ffffff")
ax.axvline(med_fatti, color="#cbd5e1", linestyle="--", linewidth=1.2)
ax.axhline(med_subiti, color="#cbd5e1", linestyle="--", linewidth=1.2)
ax.text(1.95, 0.88, "DOMINIO\n(Tanto attacco, solida difesa)", color="#1d4ed8", fontsize=8.5, fontweight="bold", ha="right")
ax.text(1.22, 1.55, "SOFFERENZA\n(Poco attacco, vulnerabili)", color="#ea580c", fontsize=8.5, fontweight="bold", ha="left")
ax.scatter(xg_fatti_gara, xg_subiti_gara, color="#1e293b", s=90, edgecolor="#ffffff", linewidth=1.5, zorder=4)
for sq, xf, xs in zip(squadre_10, xg_fatti_gara, xg_subiti_gara):
    offset_y = 0.025 if sq != "Inter" else -0.035
    ax.text(xf, xs + offset_y, sq, fontsize=9.5, fontweight="bold", color="#0f172a", ha="center")
ax.set_xlim(1.20, 2.00)
ax.set_ylim(1.60, 0.85)
ax.set_xlabel("xG Fatti a partita (Produzione offensiva attesa)", fontsize=10, fontweight="600", color="#334155")
ax.set_ylabel("xG Subiti a partita (Solidezza difensiva - Minore è meglio)", fontsize=10, fontweight="600", color="#334155")
for s in ["top", "right", "left", "bottom"]: ax.spines[s].set_visible(False)
ax.xaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax.yaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("g_slide10_quadrante_con_guida.png", dpi=250, bbox_inches="tight")
plt.close()

# Quadrante - Clean
fig, ax = plt.subplots(figsize=(10, 5), dpi=250, facecolor="#ffffff")
ax.set_facecolor("#ffffff")
ax.axvline(med_fatti, color="#cbd5e1", linestyle="--", linewidth=1.2)
ax.axhline(med_subiti, color="#cbd5e1", linestyle="--", linewidth=1.2)
ax.scatter(xg_fatti_gara, xg_subiti_gara, color="#1e293b", s=90, edgecolor="#ffffff", linewidth=1.5, zorder=4)
for sq, xf, xs in zip(squadre_10, xg_fatti_gara, xg_subiti_gara):
    offset_y = 0.025 if sq != "Inter" else -0.035
    ax.text(xf, xs + offset_y, sq, fontsize=9.5, fontweight="bold", color="#0f172a", ha="center")
ax.set_xlim(1.20, 2.00)
ax.set_ylim(1.60, 0.85)
ax.set_xlabel("xG Fatti a partita", fontsize=10, fontweight="600", color="#334155")
ax.set_ylabel("xG Subiti a partita (Meno subiti = In alto)", fontsize=10, fontweight="600", color="#334155")
for s in ["top", "right", "left", "bottom"]: ax.spines[s].set_visible(False)
ax.xaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax.yaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("g_slide10_quadrante_clean.png", dpi=250, bbox_inches="tight")
plt.close()

# 2. Slide 11: Heatmap Scostamento (Color-blind safe: Arancione = Sotto attese, Blu = Sopra attese)
colors_div = ["#ea580c", "#fed7aa", "#f8fafc", "#bfdbfe", "#1d4ed8"]
cmap_orange_blue = LinearSegmentedColormap.from_list("OrangeBlue", colors_div, N=256)

fig, ax = plt.subplots(figsize=(11, 5), dpi=250, facecolor="#ffffff")
sns.heatmap(df_diff, cmap=cmap_orange_blue, center=0, annot=True, fmt="+d", cbar=True,
            linewidths=1.2, linecolor="#ffffff", ax=ax,
            annot_kws={"fontsize": 9, "fontweight": "bold"},
            cbar_kws={"label": "Differenza (Gol Fatti - Gol Attesi)"})
for text in ax.texts:
    val = int(text.get_text())
    if val >= 8 or val <= -5:
        text.set_color("#ffffff")
    else:
        text.set_color("#0f172a")

ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9.5, fontweight="bold", color="#0f172a")
ax.set_xticklabels(ax.get_xticklabels(), fontsize=9, color="#475569")
ax.tick_params(length=0)
plt.title("Bilancio Offensivo: Gol Segnati meno Gol Attesi (xG)\n[Blu = Sopra le attese | Arancione = Sotto le attese / Gol sprecati]",
          fontsize=10.5, fontweight="bold", pad=14, color="#0f172a")
plt.tight_layout()
plt.savefig("g4_heatmap_scostamento_con_guida.png", dpi=250, bbox_inches="tight")
plt.close()

fig2, ax2 = plt.subplots(figsize=(11, 5), dpi=250, facecolor="#ffffff")
sns.heatmap(df_diff, cmap=cmap_orange_blue, center=0, annot=True, fmt="+d", cbar=True,
            linewidths=1.2, linecolor="#ffffff", ax=ax2,
            annot_kws={"fontsize": 9, "fontweight": "bold"})
for text in ax2.texts:
    val = int(text.get_text())
    if val >= 8 or val <= -5:
        text.set_color("#ffffff")
    else:
        text.set_color("#0f172a")

ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0, fontsize=9.5, fontweight="bold", color="#0f172a")
ax2.set_xticklabels(ax2.get_xticklabels(), fontsize=9, color="#475569")
ax2.tick_params(length=0)
plt.tight_layout()
plt.savefig("g4_heatmap_scostamento_clean.png", dpi=250, bbox_inches="tight")
plt.close()

# 3. Slide 11: Dumbbell Totale Decennale (Color-blind safe + Legenda esterna)
tot_gol = [sum(gol_reali[sq]) for sq in squadre]
tot_xg = [sum(xg_dict[sq]) for sq in squadre]
df_tot = pd.DataFrame({
    "squadra": squadre,
    "gol_fatti": tot_gol,
    "gol_attesi": tot_xg,
    "diff": [g - x for g, x in zip(tot_gol, tot_xg)]
}).sort_values("diff", ascending=True)
y_pos = np.arange(len(df_tot))

fig, ax = plt.subplots(figsize=(10, 5.2), dpi=250, facecolor="#ffffff")
ax.set_facecolor("#ffffff")
for y, (_, row) in enumerate(df_tot.iterrows()):
    g = row["gol_fatti"]
    xval = row["gol_attesi"]
    d = row["diff"]
    col = "#1d4ed8" if d >= 0 else "#ea580c"  # Blu per positivo, Arancione per negativo
    ax.plot([xval, g], [y, y], color=col, linewidth=4, alpha=0.35, zorder=2)
    ax.scatter(xval, y, color="#64748b", s=90, zorder=4, edgecolor="#ffffff", linewidth=1.5)
    ax.scatter(g, y, color=col, s=120, zorder=5, edgecolor="#ffffff", linewidth=2)
    sign = "+" if d >= 0 else ""
    ax.text(max(g, xval) + 12, y, f"{sign}{d} gol ({g} fatti vs {xval} attesi)",
            va="center", color=col, fontsize=9.5, fontweight="bold")

# Legenda esterna posizionata in alto senza sovrapposizioni
ax.scatter([500], [10.4], color="#64748b", s=80, clip_on=False)
ax.text(515, 10.4, "Gol che dovevano fare (xG)", va="center", color="#475569", fontsize=9.5, fontweight="600")
ax.scatter([675], [10.4], color="#1d4ed8", s=90, clip_on=False)
ax.text(690, 10.4, "Gol effettivi segnati", va="center", color="#1d4ed8", fontsize=9.5, fontweight="bold")

ax.set_yticks(y_pos)
ax.set_yticklabels(df_tot["squadra"], fontsize=10.5, fontweight="bold", color="#0f172a")
ax.set_xlim(480, 890)
ax.set_ylim(-0.8, 10.8)
ax.set_xlabel("Totale Gol nel Decennio (2014 - 2024)", fontsize=10.5, fontweight="600", color="#334155")
for s in ["top", "right", "left", "bottom"]: ax.spines[s].set_visible(False)
ax.xaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("g5_dumbbell_totale_con_guida.png", dpi=250, bbox_inches="tight")
plt.close()

fig2, ax2 = plt.subplots(figsize=(10, 5), dpi=250, facecolor="#ffffff")
ax2.set_facecolor("#ffffff")
for y, (_, row) in enumerate(df_tot.iterrows()):
    g = row["gol_fatti"]
    xval = row["gol_attesi"]
    d = row["diff"]
    col = "#1d4ed8" if d >= 0 else "#ea580c"
    ax2.plot([xval, g], [y, y], color=col, linewidth=4, alpha=0.35, zorder=2)
    ax2.scatter(xval, y, color="#64748b", s=90, zorder=4, edgecolor="#ffffff", linewidth=1.5)
    ax2.scatter(g, y, color=col, s=120, zorder=5, edgecolor="#ffffff", linewidth=2)

ax2.set_yticks(y_pos)
ax2.set_yticklabels(df_tot["squadra"], fontsize=10.5, fontweight="bold", color="#0f172a")
ax2.set_xlim(480, 800)
ax2.set_ylim(-0.8, 9.8)
ax2.set_xlabel("Totale Gol nel Decennio (2014 - 2024)", fontsize=10.5, fontweight="600", color="#334155")
for s in ["top", "right", "left", "bottom"]: ax2.spines[s].set_visible(False)
ax2.xaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax2.set_axisbelow(True)
plt.tight_layout()
plt.savefig("g5_dumbbell_totale_clean.png", dpi=250, bbox_inches="tight")
plt.close()

# 4. Slide 12: Caso Studio Punti Reali vs xPTS (2017/18, Color-blind safe)
club_caso = ["Juventus", "Napoli", "Roma", "Inter", "Lazio", "Milan", "Atalanta", "Fiorentina"]
pts_reali =  [95, 91, 77, 72, 72, 64, 60, 57]
xpts_attesi = [85, 90, 71, 70, 78, 67, 69, 58]
y_caso = np.arange(len(club_caso))

fig, ax = plt.subplots(figsize=(10, 5.2), dpi=250, facecolor="#ffffff")
ax.set_facecolor("#ffffff")
for y, (r, a) in enumerate(zip(pts_reali, xpts_attesi)):
    diff = r - a
    col = "#1d4ed8" if diff >= 0 else "#ea580c"
    ax.plot([a, r], [y, y], color=col, linewidth=3.5, alpha=0.35, zorder=2)
    ax.scatter(a, y, color="#64748b", s=80, zorder=4, edgecolor="#ffffff", linewidth=1.5)
    ax.scatter(r, y, color=col, s=110, zorder=5, edgecolor="#ffffff", linewidth=2)
    segno = "+" if diff >= 0 else ""
    ax.text(max(r, a) + 1.2, y, f"{segno}{diff} pt (Reali: {r} | Attesi: {a})",
            va="center", color=col, fontsize=9, fontweight="bold")

ax.scatter([58], [8.4], color="#64748b", s=70, clip_on=False)
ax.text(59.5, 8.4, "Punti Attesi dal gioco (xPTS)", va="center", color="#475569", fontsize=9, fontweight="600")
ax.scatter([77], [8.4], color="#1d4ed8", s=90, clip_on=False)
ax.text(78.5, 8.4, "Punti Effettivi conquistati", va="center", color="#1d4ed8", fontsize=9.5, fontweight="bold")

ax.set_yticks(y_caso)
ax.set_yticklabels(club_caso, fontsize=10, fontweight="bold", color="#0f172a")
ax.set_xlim(55, 105)
ax.set_ylim(-0.8, 8.8)
ax.set_xlabel("Punti in classifica (Stagione 2017/18: Scudetto Juve 95 vs Napoli 91)", fontsize=10, fontweight="600", color="#334155")
for s in ["top", "right", "left", "bottom"]: ax.spines[s].set_visible(False)
ax.xaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("g_slide12_caso_studio_con_guida.png", dpi=250, bbox_inches="tight")
plt.close()

fig, ax = plt.subplots(figsize=(10, 5), dpi=250, facecolor="#ffffff")
ax.set_facecolor("#ffffff")
for y, (r, a) in enumerate(zip(pts_reali, xpts_attesi)):
    diff = r - a
    col = "#1d4ed8" if diff >= 0 else "#ea580c"
    ax.plot([a, r], [y, y], color=col, linewidth=3.5, alpha=0.35, zorder=2)
    ax.scatter(a, y, color="#64748b", s=80, zorder=4, edgecolor="#ffffff", linewidth=1.5)
    ax.scatter(r, y, color=col, s=110, zorder=5, edgecolor="#ffffff", linewidth=2)

ax.set_yticks(y_caso)
ax.set_yticklabels(club_caso, fontsize=10, fontweight="bold", color="#0f172a")
ax.set_xlim(55, 100)
ax.set_ylim(-0.8, 7.8)
ax.set_xlabel("Punti in classifica (Stagione 2017/18)", fontsize=10, fontweight="600", color="#334155")
for s in ["top", "right", "left", "bottom"]: ax.spines[s].set_visible(False)
ax.xaxis.grid(True, color="#f1f5f9", linewidth=1.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("g_slide12_caso_studio_clean.png", dpi=250, bbox_inches="tight")
plt.close()

print("Tutti i grafici avanzati generati ed esportati con palette corretta (Blu/Arancione).")



Tutti i grafici avanzati generati ed esportati con palette corretta (Blu/Arancione).
